# Simple linear baselines

In [3]:
import re
import os
import random
import pandas as pd
from statsmodels.formula.api import ols
from sklearn import metrics
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()

In [2]:
def split_mut_sit(mutation):
    parts = re.match(r'([A-Za-z])(\d+)(\S)', mutation) # \S for any non- white space
    if parts:
        return list(parts.groups())
    else:
        return [None, None, None]

In [55]:
#data = pd.read_csv('../data/viral/metadata/SARS2_DELTA_SPIKE_Dadonaite.csv')
data = pd.read_csv('../data/viral/metadata/IAV_H1_HA_Wu.csv')
#data = pd.read_csv('../data/viral/metadata/SARS2_RBD_binding_Starr.csv')
data[['wt', 'site', 'mut']] = data['mutant'].apply(lambda x: pd.Series(split_mut_sit(x)))
data['target'] = scaler.fit_transform(data['target'].to_frame()).squeeze()
data['site'] = data['site'].astype(str)

common_sites = data['site'].value_counts()[data['site'].value_counts() > 4].index
data = data[data['site'].isin(common_sites)]

## Random split

In [ ]:
train_data, test_data = train_test_split(data, test_size=0.15, random_state=4)

# Fit the OLS model using formula interface
model = ols('target ~ site', data=train_data).fit()

# Make predictions on test set
y = test_data.copy()
y['target'] = 0
test_predictions = model.predict(y)

# Calculate performance metrics
test_r2 = metrics.r2_score(test_data['target'], test_predictions)

print(f"Test Set Performance:")
print(f"R² Score: {test_r2:.3f}")

In [57]:
train_data, test_data = train_test_split(data, test_size=0.2, random_state=4, stratify=data['site'])

# Fit the OLS model using formula interface
model = ols('target ~ site', data=train_data).fit()

# Make predictions on test set
y = test_data.copy()
y['target'] = 0
test_predictions = model.predict(y)

# Calculate performance metrics
test_r2 = metrics.r2_score(test_data['target'], test_predictions)

print(f"Test Set Performance:")
print(f"R² Score: {test_r2:.3f}")

Test Set Performance:
R² Score: -0.111


## Visualize results

In [62]:
import os

full_res = pd.DataFrame()
for source in ['viral', 'nonviral']:
    res_dir = f'../experiments/OLS/{source}/pool_split/'
    for file in os.listdir(res_dir):
        dataset = file.split('.csv')[0]
        df = pd.read_csv(os.path.join(res_dir, file), index_col=0)
        df['Dataset'] = dataset
        df['Source'] = source
        df['Model'] = 'OLS'
        df['Split'] = 'pooled'
        full_res = pd.concat([full_res, df])
        
full_res

,Model,Fold,R2_score_train,MAE_score_train,RMSE_score_train,R2_score_test,MAE_score_test,RMSE_score_test,rho_score_train,rho_score_test,Dataset,Source,Split
0,OLS,1,0.376406,0.628156,0.792004,0.304368,0.658004,0.824147,0.598880,0.540495,IAV_H1_NP_Doud,viral,pooled
1,OLS,2,0.379339,0.625441,0.787427,0.290447,0.672475,0.844004,0.601615,0.526581,IAV_H1_NP_Doud,viral,pooled
2,OLS,3,0.378776,0.625165,0.787603,0.300951,0.671381,0.838504,0.600995,0.533578,IAV_H1_NP_Doud,viral,pooled
0,OLS,1,0.348981,0.343840,0.762675,0.203165,0.379728,0.641082,0.639177,0.371428,IAV_H1_HA_Wu,viral,pooled
1,OLS,2,0.389340,0.328689,0.711756,-0.026567,0.398681,0.890459,0.630545,0.419289,IAV_H1_HA_Wu,viral,pooled
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1,OLS,2,0.137988,0.731342,0.926392,-0.026640,0.798818,1.022744,0.368474,0.140475,CALM1_HUMAN_Roth2017,nonviral,pooled
2,OLS,3,0.141814,0.734375,0.932268,-0.024135,0.785742,0.986381,0.369130,0.177240,CALM1_HUMAN_Roth2017,nonviral,pooled
0,OLS,1,0.509198,0.412195,0.684900,0.448029,0.479561,0.804435,0.718037,0.682826,IF1_ECOLI,nonviral,pooled
1,OLS,2,0.490962,0.425552,0.704523,0.502878,0.454994,0.738491,0.720461,0.700028,IF1_ECOLI,nonviral,pooled


In [63]:
full_res.to_csv('../experiments_v01/lassoCV/results_OLS_3folds_v_nv_Pooled.csv')

## Site split

In [39]:
def split_data(meta_data, seed, train_pct=0.8, test_pct=0.2):
    # find sites of mutation and order randomly
    meta_data["site"] = [int(s[1:-1]) for s in meta_data["mutant"]]
    sites = meta_data["site"].unique()
    random.seed(seed)
    random.shuffle(sites)

    if train_pct + test_pct != 1:
        print("Split percentages must sum to 1")
        return

    df_size = meta_data.shape[0]
    df_test_size = df_size*test_pct
    test_sites, train_sites = [], []

    # determine sites for test, then train
    for site in sites:
        if len(test_sites) <= df_test_size:
            test_sites.extend([mut_site for mut_site in meta_data["site"] if mut_site == site])
        else:
            train_sites.extend([mut_site for mut_site in meta_data["site"] if mut_site == site])

    # subset df for train, test data
    train_df = meta_data[meta_data["site"].isin(set(train_sites))]
    test_df = meta_data[meta_data["site"].isin(set(test_sites))]

    return train_df, test_df

In [ ]:
train_data, test_data = split_data(data, 4)
train_data['site'] = train_data['site'].astype(str)
test_data['site'] = test_data['site'].astype(str)

# Fit the OLS model using formula interface
model = ols('target ~ site', data=train_data).fit()

# Make predictions on test set
y = test_data.copy()
y['target'] = 0
test_predictions = model.predict(y)

# Calculate performance metrics
test_r2 = metrics.r2_score(test_data['target'], test_predictions)

print(f"Test Set Performance:")
print(f"R² Score: {test_r2:.3f}")

# Average model

In [4]:
def split_mut_sit(mutation):
    parts = re.match(r'([A-Za-z])(\d+)(\S)', mutation) # \S for any non- white space
    if parts:
        return list(parts.groups())
    else:
        return [None, None, None]

In [5]:
data = pd.read_csv('../data/viral/metadata/SARS2_BA1_SPIKE_Dadonaite.csv')
data[['wt', 'site', 'mut']] = data['mutant'].apply(lambda x: pd.Series(split_mut_sit(x)))
data

,ID,mutant,num_mutations,target,sequence,wt,site,mut
0,SARS2_BA1_M1I,M1I,1,-2.2303,IFVFLVLLPLVSSQCVNLTTRTQLPPAYTNSFTRGVYYPDKVFRSS...,M,1,I
1,SARS2_BA1_M1M,M1M,1,0.0000,MFVFLVLLPLVSSQCVNLTTRTQLPPAYTNSFTRGVYYPDKVFRSS...,M,1,M
2,SARS2_BA1_M1T,M1T,1,-2.4823,TFVFLVLLPLVSSQCVNLTTRTQLPPAYTNSFTRGVYYPDKVFRSS...,M,1,T
3,SARS2_BA1_M1V,M1V,1,-2.3068,VFVFLVLLPLVSSQCVNLTTRTQLPPAYTNSFTRGVYYPDKVFRSS...,M,1,V
4,SARS2_BA1_F2F,F2F,1,0.0000,MFVFLVLLPLVSSQCVNLTTRTQLPPAYTNSFTRGVYYPDKVFRSS...,F,2,F
...,...,...,...,...,...,...,...,...
10793,SARS2_BA1_S1249F,S1249F,1,0.3970,MFVFLVLLPLVSSQCVNLTTRTQLPPAYTNSFTRGVYYPDKVFRSS...,S,1249,F
10794,SARS2_BA1_S1249E,S1249E,1,-0.0168,MFVFLVLLPLVSSQCVNLTTRTQLPPAYTNSFTRGVYYPDKVFRSS...,S,1249,E
10795,SARS2_BA1_S1249C,S1249C,1,0.3963,MFVFLVLLPLVSSQCVNLTTRTQLPPAYTNSFTRGVYYPDKVFRSS...,S,1249,C
10796,SARS2_BA1_S1249A,S1249A,1,-0.0001,MFVFLVLLPLVSSQCVNLTTRTQLPPAYTNSFTRGVYYPDKVFRSS...,S,1249,A


In [6]:
model_avg = data.groupby(['site'])['target'].mean().to_dict()
model_avg['0'] = data['target'].mean().item()

preds = pd.DataFrame([model_avg.get(str(site), model_avg['0']) for site in data['site']])
preds

,0
0,-1.754850
1,-1.754850
2,-1.754850
3,-1.754850
4,0.139133
...,...
10793,0.201730
10794,0.201730
10795,0.201730
10796,0.201730


In [7]:
import os

full_res = pd.DataFrame()
for method in ['pool_split', 'site_split']:
    for source in ['viral', 'cellular']:
        res_dir = f'../experiments/AVG/{source}/{method}/'
        for file in os.listdir(res_dir):
            dataset = file.split('.csv')[0]
            df = pd.read_csv(os.path.join(res_dir, file), index_col=0)
            df['Dataset'] = dataset
            df['Source'] = source
            df['Model'] = 'AVG'
            if method == 'pool_split':
                df['Split'] = 'pooled'
            else:
                df['Split'] = 'site'
            full_res = pd.concat([full_res, df])
        
full_res

,Model,Fold,R2_score_train,MAE_score_train,RMSE_score_train,R2_score_test,MAE_score_test,RMSE_score_test,rho_score_train,rho_score_test,Dataset,Source,Split
0,AVG,1,0.530925,0.511884,0.683361,0.424830,0.567484,0.765066,0.734405,0.669716,SARS2_PLPRO_abundance_Wu,viral,pooled
1,AVG,2,0.521827,0.515027,0.691632,0.461210,0.551556,0.732446,0.731561,0.683581,SARS2_PLPRO_abundance_Wu,viral,pooled
2,AVG,3,0.527835,0.510688,0.683042,0.433635,0.573968,0.768914,0.733999,0.671194,SARS2_PLPRO_abundance_Wu,viral,pooled
0,AVG,1,0.632854,0.422123,0.603981,0.673508,0.412015,0.578420,0.761273,0.792606,LAMBDA_HCP_Tsuboyama,viral,pooled
1,AVG,2,0.625207,0.427129,0.611822,0.692847,0.410369,0.553779,0.770798,0.752323,LAMBDA_HCP_Tsuboyama,viral,pooled
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1,AVG,2,0.000000,0.812128,0.994095,-0.048830,0.766907,1.028071,NaN,NaN,RASH_HUMAN_Kuriyan,cellular,site
2,AVG,3,0.000000,0.774200,0.987168,-0.017285,0.852537,1.049883,NaN,NaN,RASH_HUMAN_Kuriyan,cellular,site
0,AVG,1,0.000000,0.820263,1.008093,-0.002555,0.806109,0.967561,NaN,NaN,AMIE_PSEAE_Whitehead,cellular,site
1,AVG,2,0.000000,0.814106,0.997071,-0.003854,0.846865,1.011889,NaN,NaN,AMIE_PSEAE_Whitehead,cellular,site


In [8]:
full_res.to_csv('../experiments/AVG/results_AVG_3folds_all.csv')